# EELSNMF Demo

In [1]:
import EELSNMF as enmf

cupy not available


In [2]:
# The main input for EELSNMF is hyperspy signal and the experimental conditions at which it was acquired:
import hyperspy.api as hs
s = hs.load(r"C:\Users\torruell\Documents\EPFL\eelsnmf tests\Test_SI.hspy")

# EELSNMF if a non-negative matrix factorization framework. The data need to be strictly positive:
s.data = np.clip(s.data,0,None)


# The experimental parameters are read if correctly defined in the image metadata.
deco = enmf.EELSNMF(s)
print(f"beam energy (V): {deco.E0}")
print(f"Convergence angle (rad): {deco.alpha}")
print(f"Collection angle (rad): {deco.beta}")

beam energy (V): 300000.0
Convergence angle (rad): 0.02022
Collection angle (rad): 0.036000000000000004


In [3]:
# For the physics guided approach we need to define the contributions to the measured EELS spectra, which are stored in the G matrix:

deco.build_G(
    low_loss = None, # currently the model only suports convolution with the average low-loss (not pixel-wise for a dual-EELS dataset)

    fine_structure_ranges = { #Here we define what are the edge that we expect in our EELS spectrum and their ELNES ranges (in eV)
        "Fe_L" : (704.,732.),
        "Cr_L" : (571.,597.),
        "O_K" : (526.,560.)
    }
)

deco.plot_energy_ranges()

L2 is used
L1 is used
L2 is used
L1 is used


At this stage we are ready for spectral decomposition.
There are different methods to choose depending on if smoothness and/or Sum-rule conservation want to be enforced.
The default method solves:

$$
\arg\min_{W,H}  \| X - G W H \|_F^2
$$

Using NMF multiplicative updates.

In [13]:
deco.decomposition(
    n_components = 2,
    max_iters = 500,
    tol = 1e-9, # This tolerance defines the convergence condition
    decomposition_method = "default_decomposition"
    
)

100%|████████████████████████████████████████| 500/500 [00:15<00:00, 32.04it/s, error=3.47e+6, relative change=1.05e-5]


In [22]:
#there are several built in plots to asses the results:
deco.plot_average_model() # plots the match between the average data spectrum and average model

In [23]:
deco.plot_factors() # plots the spectrum of each phase (the columns in GW)

In [24]:
deco.plot_loadings() # plots the abundance of each phase (the rows in H)

In [27]:
deco.plot_edges() # plots the ELNES associated to each element in each phase

In [26]:
deco.plot_chemical_maps()

The EELS spectrum is counting events and therefore follows Poisson statistics. A metric better suited for this case is the Kullback-Leibler divergence. The library also implements NMF to solve:

$$
\arg\min_{W,H}  D_{KL}(X\|GWH)
$$

In [20]:
deco.decomposition(
    n_components = 2,
    max_iters = 500,
    tol = 1e-9, 
    decomposition_method = "default_kl_decomposition",
    rescale_WH = True, # Fixes scale of H and W
    KL_rescaling_per_iter = True # KL divergence fits "variance", not absoulte scale. This corrects scale at each iteration (slow).
    
)

100%|█████████████████████████████████████████| 500/500 [01:46<00:00,  4.69it/s, error=2.79e+6, relative change=7.6e-6]


In [30]:
deco.decomposition(
    n_components = 2,
    max_iters = 500,
    tol = 1e-9, 
    decomposition_method = "SumRule_decomposition",
    lmbda = 1e6,
    norm = "none", #lmbda can be multiplied by the mean/elementwise G.T@X@H.T, the data weight in NMF updates, so that its easier to gauge a good value.
    SR_tolerance = 1, # sumrule penalty is not applied if its of by a factor less the SR_tolerance
    convergent_beam_correction = True,
    constrain = "both"  # penalty can be applied to cross-section coefficeints, ELNES coefficients or both.
)

C:\Users\torruell\Documents\GitHub\EELSNMF\EELSNMF\utils.py:223: RuntimeWarning: invalid value encountered in arccos
  f = (1/np.pi)*(np.arccos(x)+((beta**2)/(alpha**2))*np.arccos(y)-(1/(2*alpha**2))*sqrt)
100%|████████████████████████████████████████████████| 500/500 [00:10<00:00, 49.59it/s, error=nan, relative change=nan]


In [32]:
enmf.utils.convergent_psi(deco.energy_axis,deco.alpha,deco.beta)

C:\Users\torruell\Documents\GitHub\EELSNMF\EELSNMF\utils.py:223: RuntimeWarning: invalid value encountered in arccos
  f = (1/np.pi)*(np.arccos(x)+((beta**2)/(alpha**2))*np.arccos(y)-(1/(2*alpha**2))*sqrt)


array([nan, nan, nan, ..., nan, nan, nan])

In [51]:
thE = enmf.utils.theta_E(deco.energy_axis,300e3)

In [50]:
enmf.utils.integral_over_th(thE,deco.alpha,deco.beta,1000)

C:\Users\torruell\Documents\GitHub\EELSNMF\EELSNMF\utils.py:223: RuntimeWarning: invalid value encountered in arccos
  f = (1/np.pi)*(np.arccos(x)+((beta**2)/(alpha**2))*np.arccos(y)-(1/(2*alpha**2))*sqrt)


array([nan, nan, nan, ..., nan, nan, nan])

In [41]:
thE[0]

0.000990819842725378

In [64]:
a = 0.02
b = 0.03
enmf.utils.convergent_factor(np.linspace(0,a+b,10),a,b)

C:\Users\torruell\Documents\GitHub\EELSNMF\EELSNMF\utils.py:223: RuntimeWarning: invalid value encountered in arccos
  f = (1/np.pi)*(np.arccos(x)+((beta**2)/(alpha**2))*np.arccos(y)-(1/(2*alpha**2))*sqrt)


array([ 1.        ,  1.        ,  0.41233045,  0.46773679,  0.44961666,
        0.40311487,  0.31930358,  0.1696666 , -0.08648643,         nan])

In [ ]:
np.nan_to_num()